# 13 · Modelo panel — Regularizado (Ridge / Lasso)

Regresion regularizada sobre el panel completo (one-hot + features del tablon analitico), como contraste lineal interpretable.

- `Ridge`/`Lasso` con alpha elegido por CV (KFold 5) sobre el train
- Features numericas escaladas (StandardScaler); dummies sin escalar
- Ficha: `Ridge(alpha=CV)` / `Lasso(alpha=CV, random_state=42)`
- Resultados: `results/13_modelo_panel_regularizado_metrics.csv`


In [1]:
import polars as pl
import numpy as np
import pandas as pd
import json
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid", palette="muted")
DATA = Path("data")
RESULTS = Path("results")
MODELS = Path("../models")
RESULTS.mkdir(exist_ok=True)
MODELS.mkdir(exist_ok=True)


In [2]:
panel = pl.read_parquet(DATA / "panel_features.parquet")
meta = json.loads((DATA / "panel_metadata.json").read_text())
FEATURES = meta["features"]
TEST_START = meta["test_start"]
print("FEATURES:", FEATURES)


FEATURES: ['anio', 'iph_media', 'ocupacion_media', 'lluvia_anual_mm', 'lag1']


In [3]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def metricas(test, pred):
    test, pred = np.asarray(test, float), np.asarray(pred, float)
    return dict(
        mae=float(mean_absolute_error(test, pred)),
        mape=float(np.mean(np.abs((test - pred) / test)) * 100),
        rmse=float(mean_squared_error(test, pred) ** 0.5),
        r2=float(r2_score(test, pred)),
    )


In [4]:
from sklearn.linear_model import Ridge, Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold

pdf = panel.to_pandas()
dummies = pd.get_dummies(pdf["cod_municipio"], prefix="mun", dtype=float)
X_all = pd.concat([pdf[FEATURES], dummies], axis=1)
X_all["cod_municipio"] = pdf["cod_municipio"].values
X_all["consumo_hm3"] = pdf["consumo_hm3"].values
numeric = list(FEATURES)
feature_cols = [c for c in X_all.columns if c not in ("cod_municipio", "consumo_hm3")]

X_all_scaled = X_all.copy()
scaler = StandardScaler().fit(X_all[numeric])
X_all_scaled[numeric] = scaler.transform(X_all[numeric])
X_all_scaled["anio_raw"] = X_all["anio"].values
print("features:", len(feature_cols))


features: 72


In [5]:
train = X_all_scaled[X_all_scaled["anio_raw"] < TEST_START].dropna(subset=["lag1"])
y_train = train["consumo_hm3"].values

alphas = [0.01, 0.1, 1.0, 10.0, 100.0]
kf = KFold(n_splits=5, shuffle=True, random_state=42)

def mejor_alpha(Cls):
    scores = {}
    for a in alphas:
        errs = []
        for i_tr, i_va in kf.split(train):
            m = Cls(alpha=a, random_state=42)
            m.fit(train.iloc[i_tr][feature_cols], y_train[i_tr])
            p = m.predict(train.iloc[i_va][feature_cols])
            errs.append(np.mean(np.abs((y_train[i_va] - p) / y_train[i_va])) * 100)
        scores[a] = np.mean(errs)
    return min(scores, key=scores.get), scores

best_ridge, s_ridge = mejor_alpha(Ridge)
best_lasso, s_lasso = mejor_alpha(Lasso)
print("Ridge alphas (MAPE CV):", {k: round(v, 1) for k, v in s_ridge.items()}, "-> mejor:", best_ridge)
print("Lasso alphas (MAPE CV):", {k: round(v, 1) for k, v in s_lasso.items()}, "-> mejor:", best_lasso)

ridge = Ridge(alpha=best_ridge).fit(train[feature_cols], y_train)
lasso = Lasso(alpha=best_lasso, random_state=42).fit(train[feature_cols], y_train)


Ridge alphas (MAPE CV): {0.01: np.float64(22.4), 0.1: np.float64(17.9), 1.0: np.float64(14.7), 10.0: np.float64(18.0), 100.0: np.float64(89.4)} -> mejor: 1.0
Lasso alphas (MAPE CV): {0.01: np.float64(14.6), 0.1: np.float64(29.5), 1.0: np.float64(227.8), 10.0: np.float64(820.7), 100.0: np.float64(820.7)} -> mejor: 0.01


In [6]:
i_lag1 = numeric.index("lag1")
mu_lag1, sd_lag1 = scaler.mean_[i_lag1], scaler.scale_[i_lag1]

def predict_recursivo_lin(model, train_rows, test_rows):
    last = float(train_rows["consumo_hm3"].iloc[-1])
    preds = []
    for _, row in test_rows.iterrows():
        x = {c: row[c] for c in feature_cols}
        x["lag1"] = (last - mu_lag1) / sd_lag1  # lag1 va escalado (StandardScaler)
        p = max(float(model.predict([list(x[f] for f in feature_cols)])[0]), 0.0)
        preds.append(p)
        last = p
    return preds

filas = []
for nombre, model in [("ridge", ridge), ("lasso", lasso)]:
    for cod_t, g in panel.partition_by("cod_municipio", as_dict=True).items():
        cod = int(cod_t[0])
        test = g.filter(pl.col("anio") >= TEST_START)
        tr_p = X_all_scaled[(X_all_scaled["cod_municipio"] == cod) & (X_all_scaled["anio_raw"] < TEST_START) & (X_all_scaled["lag1"].notna())]
        te_p = X_all_scaled[(X_all_scaled["cod_municipio"] == cod) & (X_all_scaled["anio_raw"] >= TEST_START)]
        pred = predict_recursivo_lin(model, tr_p, te_p)
        filas.append({"modelo": nombre, "cod_municipio": cod,
                      **metricas(test["consumo_hm3"].to_list(), pred)})

res = pd.DataFrame(filas)
res.to_csv(RESULTS / "13_modelo_panel_regularizado_metrics.csv", index=False)
print(res.groupby("modelo")[["mae", "mape", "rmse", "r2"]].mean().round(3))


          mae    mape   rmse       r2
modelo                               
lasso   0.244  33.443  0.261 -853.690
ridge   0.180  15.625  0.197 -133.992


In [7]:
base = pd.read_csv(RESULTS / "10_baseline_metrics.csv")
naive = base[base["modelo"] == "naive"].set_index("cod_municipio")["mape"].sort_index()
ridge_m = res[res["modelo"] == "ridge"].set_index("cod_municipio")["mape"].sort_index()
print("municipios donde ridge mejora al naive:", (ridge_m < naive).sum(), "de", naive.shape[0])

coef = pd.Series(lasso.coef_, index=feature_cols)
nz = coef[coef != 0]
print("coeficientes Lasso no nulos:", len(nz), "de", len(coef))
print(nz[nz.index.str.startswith("mun_")].shape[0], "son dummies de municipio")


municipios donde ridge mejora al naive: 33 de 67
coeficientes Lasso no nulos: 3 de 72
0 son dummies de municipio


**Conclusiones**

- Contraste lineal interpretable; Lasso permite inspeccionar que municipios/features quedan activos.
- Si queda lejos de los GB, la relacion consumo-features no es lineal o el one-hot no capta los efectos municipales.
